# 01 — Data Acquisition

Descarga de las minutas del FOMC (federalreserve.gov). El sitio de la Fed tiene dos secciones:
- **Calendario actual** (`fomccalendars.htm`): reuniones recientes (2021–presente).
- **Histórico** (`fomc_historical_year.htm`): estructura de dos niveles — un índice por año
  (`fomchistoricalYYYY.htm`) que a su vez lista las minutas (PDF o HTML) de ese año.

Para cada reunión se extrae el texto plano (HTML o PDF) y se guarda como
`YYYY-MM-DD_minutes.txt`. La fecha del nombre es la de la **reunión** (no la de publicación,
~3 semanas después).

**Output:** `data/raw/minutes/` — una minuta por reunión + `index.csv` con el estado de descarga.

> El emparejado con la federal funds target rate (FRED) y la construcción de etiquetas se hace en
> `02_preprocessing.ipynb`. El filtrado al rango de años usado para modelar también vive ahí: este
> notebook baja **todo** lo disponible.

## 1. Configuración e imports

In [ ]:
import io
import re
import csv
import time
import logging
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pdfplumber

# Rutas del repo (las minutas van a data/raw/minutes/, no a una carpeta suelta)
BASE_DIR   = Path('..').resolve()
OUTPUT_DIR = BASE_DIR / 'data' / 'raw' / 'minutes'
INDEX_FILE = OUTPUT_DIR / 'index.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Endpoints de la Fed
BASE_URL       = 'https://www.federalreserve.gov'
CURRENT_URL    = f'{BASE_URL}/monetarypolicy/fomccalendars.htm'
HIST_INDEX_URL = f'{BASE_URL}/monetarypolicy/fomc_historical_year.htm'

REQUEST_DELAY = 1.2  # segundos entre requests (cortesía con el servidor)
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; FOMC-Research-Scraper/2.0)'}

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-7s  %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('fomc')
log.info('PDF backend: pdfplumber | salida: %s', OUTPUT_DIR)

## 2. Helpers de descarga y extracción de texto

Funciones para bajar una URL y extraer texto plano sea HTML o PDF, más utilidades de limpieza y
de inferencia de la fecha a partir de la URL (`fomcminutes20230201.htm` → `2023-02-01`).

In [ ]:
def get_soup(url):
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return BeautifulSoup(resp.text, 'lxml')

def fetch_bytes(url):
    """Devuelve (content_bytes, content_type)."""
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return resp.content, resp.headers.get('Content-Type', '')

def extract_text_from_pdf(pdf_bytes):
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        pages = [p.extract_text().strip() for p in pdf.pages if p.extract_text()]
    return '\n\n'.join(pages)

def extract_text_from_html(html_bytes):
    soup = BeautifulSoup(html_bytes, 'lxml')
    for selector in ['div#article', 'div.col-xs-12.col-sm-8.col-md-8', 'div#content',
                     'div.text', 'article', 'main']:
        container = soup.select_one(selector)
        if container:
            return container.get_text(separator='\n').strip()
    return soup.get_text(separator='\n').strip()  # fallback: body entero

def fetch_and_extract(url):
    """Baja una URL y devuelve texto plano sea cual sea el formato."""
    raw_bytes, content_type = fetch_bytes(url)
    ct = content_type.lower()
    if 'pdf' in ct or url.lower().endswith('.pdf'):
        log.info('    -> PDF detectado, extrayendo texto...')
        return extract_text_from_pdf(raw_bytes)
    if 'text/plain' in ct:
        return raw_bytes.decode('utf-8', errors='replace')
    return extract_text_from_html(raw_bytes)

def clean_text(text):
    """Colapsa líneas en blanco excesivas."""
    cleaned, blank = [], 0
    for line in text.splitlines():
        if line.strip() == '':
            blank += 1
            if blank <= 2:
                cleaned.append('')
        else:
            blank = 0
            cleaned.append(line)
    return '\n'.join(cleaned).strip()

def infer_date_from_url(url):
    m = re.search(r'(\d{8})', url)
    if m:
        try:
            return datetime.strptime(m.group(1), '%Y%m%d').strftime('%Y-%m-%d')
        except ValueError:
            return m.group(1)
    return 'unknown'

def safe_filename(date_str):
    return f'{date_str}_minutes.txt'

## 3. Descubrimiento de links

Recorre el calendario actual y, vía la navegación de dos niveles, las páginas históricas por
año, juntando todas las URLs de minutas (PDF o HTML).

In [ ]:
def find_minutes_links_on_page(soup, page_url):
    """Links que apuntan a minutas reales (fomcminutesYYYYMMDD.pdf/.htm)."""
    results, seen = [], set()
    for a in soup.find_all('a', href=True):
        href = a['href']; href_lower = href.lower()
        text_lower = a.get_text(strip=True).lower()
        is_minutes = ('fomcminutes' in href_lower or 'minutes' in text_lower
                      or bool(re.search(r'minutes\d{8}', href_lower)))
        if not is_minutes:
            continue
        # Excluir links a otra página-índice de año (no son minutas)
        if 'fomchistorical' in href_lower and 'minutes' not in href_lower:
            continue
        full_url = urljoin(page_url, href)
        if full_url in seen:
            continue
        seen.add(full_url)
        results.append({'date': infer_date_from_url(full_url), 'url': full_url})
    return results

def discover_historical_year_urls(soup):
    """Del índice fomc_historical_year.htm, junta las páginas por año fomchistoricalYYYY.htm."""
    year_urls, seen = [], set()
    for a in soup.find_all('a', href=True):
        if re.search(r'fomchistorical\d{4}\.htm', a['href'], re.IGNORECASE):
            full = urljoin(HIST_INDEX_URL, a['href'])
            if full not in seen:
                seen.add(full); year_urls.append(full)
    return sorted(year_urls)

def discover_all_links():
    all_links = []
    # 1. Reuniones recientes (2021–presente)
    log.info('Calendario actual: %s', CURRENT_URL)
    links = find_minutes_links_on_page(get_soup(CURRENT_URL), CURRENT_URL)
    log.info('  %d links en el calendario actual.', len(links))
    all_links.extend(links)
    # 2. Histórico (1936–2020) vía navegación de dos niveles
    log.info('Índice histórico: %s', HIST_INDEX_URL)
    year_urls = discover_historical_year_urls(get_soup(HIST_INDEX_URL))
    log.info('  %d páginas históricas por año.', len(year_urls))
    for year_url in year_urls:
        m = re.search(r'fomchistorical(\d{4})\.htm', year_url, re.IGNORECASE)
        log.info('  Año %s ...', m.group(1) if m else year_url)
        try:
            links = find_minutes_links_on_page(get_soup(year_url), year_url)
            log.info('    -> %d links', len(links))
            all_links.extend(links)
        except Exception as exc:
            log.warning('    No se pudo bajar %s: %s', year_url, exc)
    # Deduplicar por URL y ordenar cronológicamente
    seen, unique = set(), []
    for item in all_links:
        if item['url'] not in seen:
            seen.add(item['url']); unique.append(item)
    unique.sort(key=lambda x: x['date'])
    log.info('Total de links únicos: %d', len(unique))
    return unique

## 4. Descarga

Baja cada minuta a `data/raw/minutes/`. **Idempotente:** si el archivo ya existe, lo saltea
(por eso re-correr para sumar años nuevos solo descarga lo que falta). Escribe un `index.csv`
con el estado de cada descarga.

In [ ]:
def download_all(links):
    rows = []
    for i, item in enumerate(links, 1):
        filename = safe_filename(item['date'])
        filepath = OUTPUT_DIR / filename
        log.info('[%d/%d] %s -> %s', i, len(links), item['date'], filename)
        if filepath.exists():
            log.info('  Ya existe, salteando.')
            rows.append({**item, 'local_file': filename, 'status': 'skipped'})
            continue
        try:
            clean = clean_text(fetch_and_extract(item['url']))
            if len(clean) < 100:
                raise ValueError(f'Texto demasiado corto ({len(clean)} chars) — extracción falló')
            filepath.write_text(clean, encoding='utf-8')
            rows.append({**item, 'local_file': filename, 'status': 'ok'})
            log.info('  Guardado (%d chars)', len(clean))
        except Exception as exc:
            log.error('  FALLO: %s', exc)
            rows.append({**item, 'local_file': '', 'status': f'error: {exc}'})
    with INDEX_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['date', 'url', 'local_file', 'status'])
        writer.writeheader(); writer.writerows(rows)
    ok    = sum(1 for r in rows if r['status'] == 'ok')
    skip  = sum(1 for r in rows if r['status'] == 'skipped')
    error = sum(1 for r in rows if r['status'].startswith('error'))
    log.info('Listo. Descargadas: %d | Salteadas: %d | Errores: %d', ok, skip, error)
    log.info('Índice en: %s', INDEX_FILE)

## 5. Ejecutar

Corre el pipeline completo. Tarda varios minutos (≈1.2 s por request, cientos de documentos).
Como es idempotente, re-ejecutar solo trae las minutas nuevas (p. ej. años recientes).

In [ ]:
links = discover_all_links()
download_all(links)